# Создание списка слов русского языка на основе данных Wiktionary

## Оглавление

* [Введение](#id1)
* [Подготовка рабочего пространства](#id2)
* [Создание словаря на основе названий статей](#id3)
* [Создание словаря на основе полного дампа содержимого русского Викисловаря](#id4)
* [Сравнение с pymorphy3](#id5)
  * [Оценка на основе текста А.С. Пушкина "Евгений Онегин"](#id6)
  * [Оценка на основе текстов песен Noize MC *](#id7)
* [Вывод](#id8)

--------
\* ПРИЗНАН МИНЮСТОМ РФ ИНОАГЕНТОМ

<a id="id1"></a>
## Введение

В задачах обработки естественного языка (NLP) часто требуется словарь русского языка, содержащий полный список русских слов. 

Имеется общепринятый словарь pymorphy3, созданный на базе OpenCorpora (крупного корпуса русскоязычных текстов). Однако этот словарь охватывает не все слова русского языка. 

Я решил собрать собственный словарь русских слов, который будет дополнять словарь pymorphy3, и который буду использовать для базовой проверки корректрости слов в тексте. Под словарём я имею ввиду именно список слов, а не полноценный лингвистический словарь с определениями и разметкой.

Такой словарь можно получить на основе данных сайта Wiktionary (Викисловарь).

Wiktionary - это открытый многоязычный словарь, в котором каждая словарная единица представлена в виде отдельной статьи. Викисловарь содержит информацию о словах различных языков, включая русский, а также сведения об их значениях, грамматических характеристиках и формах.

Для работы с данными проектов Wikimedia предоставляются регулярные дампы, публикуемые на https://dumps.wikimedia.org/.<br>
Дамп представляет собой архив с полной или частичной выгрузкой содержимого сайта в структурированном виде, пригодном для автоматизированной обработки.

<a id="id2"></a>
## Подготовка рабочего пространства

In [1]:
import gzip
import re
from pathlib import Path
import pandas as pd
import bz2
import shutil
import xml.etree.ElementTree as ET
from pymorphy3 import MorphAnalyzer

RANDOM_STATE = 42

<a id="id3"></a>
## Создание словаря на основе названий статей

Скачаем файл `ruwiktionary-latest-all-titles-in-ns0.gz` по ссылке https://dumps.wikimedia.org/ruwiktionary/latest/ruwiktionary-latest-all-titles-in-ns0.gz.<br>
Файл содержит список заголовков всех статей из основного пространства имён русского раздела Wiktionary.

Выберем только статьи, названия которых состоят из букв русского алфавита, дефисов и апострофов (для слов типа д'артаньян).<br>
Список из таких названий статей как раз и будет словарем русского языка.

In [2]:
input_file = Path("ruwiktionary-latest-all-titles-in-ns0.gz")
output_csv = Path("ruwiktionary_russian_words.csv")
output_txt = Path("ruwiktionary_russian_words.txt")

In [3]:
# Русское слово: кириллица, внутри допускаются дефисы и апострофы
ru_pattern = re.compile(r"^[А-Яа-яЁё]+(?:[-'][А-Яа-яЁё]+)*$")

words = set()

with gzip.open(input_file, "rt", encoding="utf-8") as file:
    for line in file:
        word = line.strip()

        # Оставляем только русские слова
        if ru_pattern.fullmatch(word):
            words.add(word.lower())

words = sorted(words)

df = pd.DataFrame(words, columns=["word"])

df.to_csv(output_csv, index=False, encoding="utf-8-sig")

with open(output_txt, "w", encoding="utf-8") as file:
    for word in words:
        file.write(word + "\n")

print(f"Готово. Найдено слов: {len(words)}")
print(f"CSV сохранён: {output_csv}")
print(f"TXT сохранён: {output_txt}")

Готово. Найдено слов: 1542954
CSV сохранён: ruwiktionary_russian_words.csv
TXT сохранён: ruwiktionary_russian_words.txt


Выведем первые строки словаря.

In [4]:
df.head(10)

,word
0,а
1,а'асх
2,а'атчевматальын
3,а'атчегматальын
4,а'к'алтывагыргын
5,а-а
6,а-а-а
7,а-а-а-а
8,а-ах
9,а-во


В таблице есть ненастоящие слова. Видимо в выборку попадают все статьи с кириллицей, а не только на русском языке. Также в выборке, возможно, есть ошибочные написания слов, страницы которых переводят на реальные слова. 

Для повышения качества словаря попробуем другой, более полный дамп, где будет возможность дополнительной фильтрации слов.

<a id="id4"></a>
## Создание словаря на основе полного дампа содержимого русского Викисловаря

Скачаем файл `ruwiktionary-latest-pages-articles-multistream.xml.bz2` по ссылке https://dumps.wikimedia.org/ruwiktionary/latest/ruwiktionary-latest-pages-articles-multistream.xml.bz2.<br>
Файл содержит тексты всех статей русского раздела Wiktionary. 

Wiktionary является многоязычным ресурсом, одна статья может содержать разделы для нескольких языков. Для выделения русскоязычной части статьи используется wiki-шаблон {{-ru-}}, который обозначает начало раздела русского языка внутри статьи.

Фильтрация с помощью {{-ru-}} позволит отбросить пустые страницы и страницы, написанные на кириллице, но не имеющие русскоязычного раздела.

Распакуем архив.

In [5]:
input_bz2 = Path("ruwiktionary-latest-pages-articles-multistream.xml.bz2")
output_xml = Path("ruwiktionary-latest-pages-articles-multistream.xml")

with bz2.open(input_bz2, "rb") as src, open(output_xml, "wb") as dst:
    shutil.copyfileobj(src, dst, length=1024 * 1024 * 16)

print("Распаковка завершена")

Распаковка завершена


Выберем статьи с русским разделом.

In [6]:
input_xml = Path("ruwiktionary-latest-pages-articles-multistream.xml")
output_txt = Path("russian_dictionary.txt")

ru_pattern = re.compile(r"^[А-Яа-яЁё]+(?:[-'][А-Яа-яЁё]+)*$")

bad_patterns = [
    re.compile(r"^(?![авикосуя]$)[а-яё]$"),        # убираем односимвольные слова кроме а, в, и, к, о, с, у, я
    re.compile(r"^(а|о|у|э|и|ы|е|я|ё|ю)(-\1)+$"),  # убираем шум, крики, припевки типа а-а-а
]

words = set()
page_count = 0

with open(input_xml, "r", encoding="utf-8") as f:
    context = ET.iterparse(f, events=("end",))

    for event, elem in context:
        if elem.tag.endswith("page"):
            page_count += 1

            title = elem.find("./{*}title")
            text = elem.find(".//{*}text")

            if title is not None and text is not None:
                word = (title.text or "").strip().lower()
                article_text = text.text or ""

                if "{{-ru-}}" in article_text:
                    if (
                        ru_pattern.fullmatch(word)
                        and not any(p.fullmatch(word) for p in bad_patterns)
                    ):
                        words.add(word)

            if page_count % 100000 == 0:
                print(f"Обработано страниц: {page_count:,}, найдено слов: {len(words):,}")

            elem.clear()

with open(output_txt, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(words)))

print(f"Готово. Найдено слов: {len(words):,}")

Обработано страниц: 100,000, найдено слов: 5,421
Обработано страниц: 200,000, найдено слов: 57,430
Обработано страниц: 300,000, найдено слов: 122,536
Обработано страниц: 400,000, найдено слов: 124,940
Обработано страниц: 500,000, найдено слов: 125,854
Обработано страниц: 600,000, найдено слов: 149,921
Обработано страниц: 700,000, найдено слов: 156,096
Обработано страниц: 800,000, найдено слов: 162,552
Обработано страниц: 900,000, найдено слов: 205,546
Обработано страниц: 1,000,000, найдено слов: 215,474
Обработано страниц: 1,100,000, найдено слов: 249,555
Обработано страниц: 1,200,000, найдено слов: 293,223
Обработано страниц: 1,300,000, найдено слов: 323,535
Обработано страниц: 1,400,000, найдено слов: 405,248
Обработано страниц: 1,500,000, найдено слов: 414,217
Обработано страниц: 1,600,000, найдено слов: 414,505
Обработано страниц: 1,700,000, найдено слов: 415,578
Обработано страниц: 1,800,000, найдено слов: 415,683
Обработано страниц: 1,900,000, найдено слов: 415,734
Обработано стр

Сохраним в CSV

In [7]:
input_txt = Path("russian_dictionary.txt")
output_csv = Path("russian_dictionary.csv")

with open(input_txt, "r", encoding="utf-8") as f:
    words = [line.strip() for line in f if line.strip()]

df = pd.DataFrame(words, columns=["word"])
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print('Размер словаря:', len(df), 'слов')

Размер словаря: 444148 слов


Выведем первые строки словаря.

In [8]:
df.head(10)

,word
0,а
1,а-во
2,а-дато
3,а-каприччио
4,а-конто
5,а-ля
6,а-ля-карт
7,а-мольный
8,а-форфе
9,аа


Выведем случайные строки словаря.

In [9]:
df.sample(10, random_state=42)

,word
163564,лемго
231463,орзали
23402,бацкино
400904,укиха
235468,ответчица
238978,отнимавши
206539,неоязычество
439786,энрике
377324,сторновав
233787,остекление


Составим таблицу со значениями выбранных слов и оценим качество и пригодность словаря.

| Слово               | Значение                                                                   | Ссылка                                                                                                   |
| ------------------- | -------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| а                   | союз; противопоставление, сопоставление		                           | [https://ru.wiktionary.org/wiki/а](https://ru.wiktionary.org/wiki/а)                               |
| а-во                | сокращение от агентство			                           | [https://ru.wiktionary.org/wiki/а-во](https://ru.wiktionary.org/wiki/а-во)                               |
| а-дато              | финансовый термин: число, от которого дан тот или иной документ     	   | [https://ru.wiktionary.org/wiki/а-дато](https://ru.wiktionary.org/wiki/а-дато)                           |
| а-каприччио         | музыкальный термин: произвольно, не подчиняясь установившимся формам       | [https://ru.wiktionary.org/wiki/а-каприччио](https://ru.wiktionary.org/wiki/а-каприччио)                 |
| а-конто             | финансовый термин: вид предварительного расчёта, представляющий собой авансовую оплату счетов экспортёра импортёром за проданные товары  | [https://ru.wiktionary.org/wiki/а-конто](https://ru.wiktionary.org/wiki/а-конто)                         |
| а-ля                | подобно, наподобие, словно, по образцу                                                        | [https://ru.wiktionary.org/wiki/а-ля](https://ru.wiktionary.org/wiki/а-ля)                               |
| а-ля-карт    | по меню, по карте                                                             | [https://ru.wiktionary.org/wiki/а-ля-карт](https://ru.wiktionary.org/wiki/а-ля-карт)       |
| а-мольный           | написанный в тональности ля минор                                          | [https://ru.wiktionary.org/wiki/а-мольный](https://ru.wiktionary.org/wiki/а-мольный)                     |
| а-форфе             | заранее произведённая налоговыми органами оценка прибыли предприятия, на котором не ведётся точный бухгалтерский учёт  | [https://ru.wiktionary.org/wiki/а-форфе](https://ru.wiktionary.org/wiki/а-форфе)  |
| аа                  | междометие (эмоциональный звук)                                            | [https://ru.wiktionary.org/wiki/аа](https://ru.wiktionary.org/wiki/аа)                                   |
| лемго        | город в Германии                                              | [https://ru.wiktionary.org/wiki/Лемго](https://ru.wiktionary.org/wiki/Лемго)               |
| орзали       | мужское имя                                                                   | [https://ru.wiktionary.org/wiki/Орзали](https://ru.wiktionary.org/wiki/Орзали)             |
| бацкино      | село в России                                                     | [https://ru.wiktionary.org/wiki/Бацкино](https://ru.wiktionary.org/wiki/Бацкино)           |
| укиха        | город в Японии                                                       | [https://ru.wiktionary.org/wiki/Укиха](https://ru.wiktionary.org/wiki/Укиха)               |
| ответчица    | женск. к ответчик                                                        | [https://ru.wiktionary.org/wiki/ответчица](https://ru.wiktionary.org/wiki/ответчица)       |
| отнимавши    | дееприч. от отнимать                                                     | [https://ru.wiktionary.org/wiki/отнимавши](https://ru.wiktionary.org/wiki/отнимавши)       |
| неоязычество | новое язычество; движение, связанное с возрождением доавраамических верований | [https://ru.wiktionary.org/wiki/неоязычество](https://ru.wiktionary.org/wiki/неоязычество) |
| энрике       | мужское имя                                                                   | [https://ru.wiktionary.org/wiki/Энрике](https://ru.wiktionary.org/wiki/Энрике)             |
| сторновав    | дееприч. от сторновать                                                       | [https://ru.wiktionary.org/wiki/сторновав](https://ru.wiktionary.org/wiki/сторновав)       |
| остекление   | действие по значению гл. остеклить                                      | [https://ru.wiktionary.org/wiki/остекление](https://ru.wiktionary.org/wiki/остекление)     |



Из таблицы видно, что большинство слов действительно имеют значение и употребляются в русскоязычных текстах. В словаре встречаются междометия, которые не несут большого смыслового значения, но тем не менее употребляются в речи. Также в словаре есть топонимы, имена и фамилии, много редко используемых, устаревших или узкоспециализированных слов, что естественно ввиду большого размера словаря.

Важно отметить, что полученный словарь не лемматизирован и включает как леммы, так и различные словоформы (например, причастия и деепричастия).

<a id="id5"></a>
## Сравнение с pymorphy3

Получим словарь библиотеки pymorphy3.

In [10]:
morph = MorphAnalyzer()

words = list(morph.dictionary.words.keys())
print("Количество словоформ в словаре pymorphy3:", len(words))

Количество словоформ в словаре pymorphy3: 5140211


In [11]:
ru_pattern = re.compile(r"^[А-Яа-яЁё]+(?:[-'][А-Яа-яЁё]+)*$")

filtered_words = [w for w in words if ru_pattern.fullmatch(w)]

df = pd.DataFrame({"word": filtered_words})

# сохраним словарь pymorphy3
df.to_csv("pymorphy_dictionary.csv", index=False, encoding="utf-8-sig")

print("После фильтра:", len(filtered_words))

После фильтра: 5137805


Прочитаем оба словаря

In [12]:
pymorphy_dictionary = pd.read_csv("pymorphy_dictionary.csv")      # словарь pymorphy
russian_dictionary = pd.read_csv("russian_dictionary.csv")        # мой словарь

# привести к одному виду
words_pymorphy_dictionary = set(pymorphy_dictionary["word"].astype(str).str.lower().str.strip())
words_russian_dictionary = set(russian_dictionary["word"].astype(str).str.lower().str.strip())

**Сравним словари**

In [13]:
only_in_pymorphy = words_pymorphy_dictionary - words_russian_dictionary

In [14]:
only_in_wiktionary = words_russian_dictionary - words_pymorphy_dictionary

In [15]:
common = words_pymorphy_dictionary & words_russian_dictionary

Сохраним слова которые встречаются только в словаре из wiktionary, но отсутствуют в pymorphy.

In [16]:
df_only_in_wiktionary = pd.DataFrame({"word": sorted(only_in_wiktionary)})
df_only_in_wiktionary.to_csv("only_in_wiktionary.csv", index=False, encoding="utf-8-sig")

In [17]:
with open("only_in_wiktionary.txt", "w", encoding="utf-8") as f:
    for w in sorted(only_in_wiktionary):
        f.write(w + "\n")

Примеры слов, которые встречаются только в pymorphy:

In [18]:
print(sorted(only_in_pymorphy)[:10])

['а-а-а', 'а-а-а-а', 'а-осей', 'а-оси', 'а-ось', 'а-осью', 'а-осям', 'а-осями', 'а-осях', 'ааво']


Примеры слов, которые встречаются только в wiktionary:

In [19]:
print(sorted(only_in_wiktionary)[:10])

['а-во', 'а-дато', 'а-каприччио', 'а-ля-карт', 'а-мольный', 'а-форфе', 'аав', 'аазаз', 'ааиса', 'ааиша']


Примеры общих слов:

In [20]:
print(sorted(common)[:10])

['а', 'а-конто', 'а-ля', 'аа', 'ааа', 'аарон', 'ааронов', 'аахенский', 'аб', 'аба']


In [21]:
print("Всего pymorphy:", len(words_pymorphy_dictionary))
print("Всего wiktionary:", len(words_russian_dictionary))

print("Только pymorphy:", len(only_in_pymorphy))
print("Только wiktionary:", len(only_in_wiktionary))
print("Общие:", len(common))

Всего pymorphy: 3063823
Всего wiktionary: 444148
Только pymorphy: 2814668
Только wiktionary: 194993
Общие: 249155


In [22]:
print(f"Покрытие pymorphy моим словарём: {100*len(common)/len(words_pymorphy_dictionary):.2}%")
print(f"Покрытие моего словаря pymorphy: {100*len(common)/len(words_russian_dictionary):.3}%")
print(f"Уникальные слова в Wiktionary: {100*len(only_in_wiktionary)/len(words_russian_dictionary):.3}%")

Покрытие pymorphy моим словарём: 8.1%
Покрытие моего словаря pymorphy: 56.1%
Уникальные слова в Wiktionary: 43.9%


**Итог:** <br>
Словарь pymorphy смеет более 3 млн. словоформ, против 444 тыс. слов в моем словаре.<br>
Примерно 43,9% моего словаря состоит из уникальных слов, которых нет в словаре pymorphy, что может быть полезным при анализет текстов. 

<a id="id6"></a>
## Оценка на основе текста А.С. Пушкина "Евгений Онегин"

Прочитаем файл с текстом и посчитаем все уникальные слова.

In [23]:
morph = MorphAnalyzer()

with open("Пушкин Александр. Евгений Онегин.txt", "r", encoding="utf-8") as f:
    text = f.read()

# приводим к нижнему регистру
text = text.lower()

# удаляем текст в квадратных скобках
text = re.sub(r"\[.*?\]", "", text)

# паттерн слова (кириллица + дефис + апостроф)
word_pattern = re.compile(r"[а-яё]+(?:[-'][а-яё]+)*")

# извлекаем слова
onegin_words = set(word_pattern.findall(text))

print("Всего уникальных словоформ в тексте:", len(onegin_words))

Всего уникальных словоформ в тексте: 9333


Посмотрип покрытие текста словарем pymorphy.

In [24]:
def is_known(word):
    return any(p.is_known for p in morph.parse(word))

known_words = {w for w in onegin_words if is_known(w)}
unknown_words = onegin_words - known_words

print("Известны pymorphy:", len(known_words))
print("Неизвестны pymorphy:", len(unknown_words))

Известны pymorphy: 9038
Неизвестны pymorphy: 295


Лемматизируем только слова, которые pymorphy знает

In [25]:
onegin_words_lemmatized = set()

for word in onegin_words:
    parses = morph.parse(word)
    
    if any(p.is_known for p in parses):
        # берем первый известный разбор
        known_parse = next(p for p in parses if p.is_known)
        onegin_words_lemmatized.add(known_parse.normal_form)
    else:
        # неизвестные слова оставляем как есть
        onegin_words_lemmatized.add(word)


known_words_lemmatized = {
    w for w in onegin_words_lemmatized
    if any(p.is_known for p in morph.parse(w))
}

unknown_words = onegin_words_lemmatized - known_words_lemmatized


print("Всего уникальных лемм и неизвестных словоформ:", len(onegin_words_lemmatized))
print("Известны pymorphy:", len(known_words_lemmatized))
print("Неизвестны pymorphy:", len(unknown_words))

Всего уникальных лемм и неизвестных словоформ: 5519
Известны pymorphy: 5224
Неизвестны pymorphy: 295


In [26]:
print(f"Доля неизвестных словоформ для словаря pymorphy: {len(unknown_words)/len(onegin_words_lemmatized)*100:.2}%")

Доля неизвестных словоформ для словаря pymorphy: 5.3%


In [27]:
print(sorted(unknown_words)[:10])

['авзонии', 'автомедоны', 'акулькой', 'альбана', 'апулей', 'апулея', 'арагвы', 'атридом', 'балтическим', 'безделие']


**Пересечение unknown слов с моим словарём**

In [28]:
russian_dict_set = set(
    russian_dictionary["word"].astype(str).str.lower().str.strip()
)

covered_by_wiki = unknown_words & russian_dict_set

print("Из unknown покрыты Wiktionary:", len(covered_by_wiki))

Из unknown покрыты Wiktionary: 43


In [29]:
print(sorted(covered_by_wiki))

['безделие', 'васисдас', 'вертер', 'веспер', 'встретя', 'гименей', 'дале', 'дальный', 'два-три', 'двурогий', 'дельвиг', 'заране', 'зизи', 'измлада', 'катенин', 'киприда', 'княжнин', 'митридат', 'морфей', 'наруже', 'нашед', 'незапный', 'облак', 'оне', 'оставя', 'осьмнадцать', 'осьмой', 'охтенка', 'парис', 'пашенька', 'по-змеиному', 'полураскрытый', 'почуя', 'принесть', 'расин', 'татьянин', 'устар', 'ушед', 'филомела', 'хлеб-соль', 'хлопание', 'чадаев', 'шишков']


In [30]:
print(f"Мой словарь дополнительно покрыл: {len(covered_by_wiki)/len(onegin_words_lemmatized)*100:.2}% текста")

Мой словарь дополнительно покрыл: 0.78% текста


Финальный остаток

In [31]:
final_unknown = unknown_words - russian_dict_set

print("Остались после обоих словарей:", len(final_unknown))

Остались после обоих словарей: 252


In [32]:
final_unknown
print(sorted(final_unknown)[:10])

['авзонии', 'автомедоны', 'акулькой', 'альбана', 'апулей', 'апулея', 'арагвы', 'атридом', 'балтическим', 'бентама']


На примере «Евгения Онегина» словарь на основе данных Wiktionary дополнительно нашёл часть слов, отсутствующих в pymorphy. Среди них есть имена собственные, фамилии, топонимы, архаичные формы, устаревшая лексика и устойчивые сочетания с дефисом.

Дополнительное покрытие составило 0,78% уникальных словоформ текста. На первый взгляд это немного, но даже небольшое дополнительное покрытие полезно: оно уменьшает количество слов, которые нужно проверять вручную.

<a id="id7"></a>
## Оценка на основе текстов песен Noize MC

Прочитаем файл с текстами песен и посчитаем все уникальные слова.

In [33]:
morph = MorphAnalyzer()

with open("Noize MC all texts.txt", "r", encoding="utf-8") as f:
    text = f.read()

# приводим к нижнему регистру
text = text.lower()

# удаляем текст в квадратных скобках
text = re.sub(r"\[.*?\]", "", text)

# паттерн слова (кириллица + дефис + апостроф)
word_pattern = re.compile(r"[а-яё]+(?:[-'][а-яё]+)*")

# извлекаем слова
noize_mc_words = set(word_pattern.findall(text))

print("Всего уникальных словоформ в тексте:", len(noize_mc_words))

Всего уникальных словоформ в тексте: 23365


Посмотрип покрытие текста словарем pymorphy.

In [34]:
def is_known(word):
    return any(p.is_known for p in morph.parse(word))

known_words = {w for w in noize_mc_words if is_known(w)}
unknown_words = noize_mc_words - known_words

print("Известны pymorphy:", len(known_words))
print("Неизвестны pymorphy:", len(unknown_words))

Известны pymorphy: 21566
Неизвестны pymorphy: 1799


Лемматизируем только слова, которые pymorphy знает.

In [35]:
noize_mc_words_lemmatized = set()

for word in noize_mc_words:
    parses = morph.parse(word)
    
    if any(p.is_known for p in parses):
        # берем первый известный разбор
        known_parse = next(p for p in parses if p.is_known)
        noize_mc_words_lemmatized.add(known_parse.normal_form)
    else:
        # неизвестные слова оставляем как есть
        noize_mc_words_lemmatized.add(word)


known_words_lemmatized = {
    w for w in noize_mc_words_lemmatized
    if any(p.is_known for p in morph.parse(w))
}

unknown_words = noize_mc_words_lemmatized - known_words_lemmatized


print("Всего уникальных лемм и неизвестных словоформ:", len(noize_mc_words_lemmatized))
print("Известны pymorphy:", len(known_words_lemmatized))
print("Неизвестны pymorphy:", len(unknown_words))

Всего уникальных лемм и неизвестных словоформ: 13481
Известны pymorphy: 11682
Неизвестны pymorphy: 1799


In [36]:
print(f"Доля неизвестных словоформ для словаря pymorphy: {len(unknown_words)/len(noize_mc_words_lemmatized)*100:.3}%")

Доля неизвестных словоформ для словаря pymorphy: 13.3%


In [37]:
print(sorted(unknown_words)[:10])

['а-а', 'а-а-а-а-а-а-а-а-а', 'а-ай', 'абырвалг', 'авиарежим', 'авиарежиме', 'автографы-то', 'автопати', 'автотюна', 'автотюном']


**Пересечение unknown слов с моим словарём**

In [38]:
russian_dict_set = set(
    russian_dictionary["word"].astype(str).str.lower().str.strip()
)

covered_by_wiki = unknown_words & russian_dict_set

print("Из unknown покрыты Wiktionary:", len(covered_by_wiki))

Из unknown покрыты Wiktionary: 449


In [39]:
print(sorted(covered_by_wiki))

['абырвалг', 'авиарежим', 'адуха', 'ак', 'алко', 'алкотестер', 'арал', 'аренби', 'ариведерчи', 'аспартам', 'атс', 'ахтунг', 'бабл-гам', 'басс', 'батл', 'баттл', 'баттлить', 'баттхёрт', 'башли', 'бело-сине-красный', 'беретта', 'беспонтовый', 'бигфут', 'бисексуал', 'биткоин', 'бла', 'бла-бла-бла', 'блатняк', 'блеванув', 'блогинг', 'ботать', 'бошка', 'братуха', 'бро', 'бу', 'буль', 'бульбулятор', 'бумбокс', 'бумер', 'бэ', 'бэтмобиль', 'бэха', 'ванёк', 'вебкамщица', 'веган', 'вегас', 'век-волкодав', 'вещдок', 'видос', 'вмазаться', 'впаривать', 'впарить', 'вуду', 'вусмерть', 'въёбывать', 'выбешивать', 'выебать', 'вых', 'вышак', 'выёбываться', 'гаврош', 'галимый', 'ган', 'ганджубас', 'гелика', 'гемор', 'гербалайф', 'гифка', 'гля', 'говнище', 'говнорок', 'годзилла', 'гоморра', 'гондон', 'гонево', 'гопота', 'госпел', 'готичный', 'грэмми', 'гэбня', 'да-да', 'давай-ка', 'далёко', 'два-три', 'движ', 'двуствольный', 'ддт', 'делирий', 'демо', 'дестрой', 'децл', 'джа', 'джангл', 'дисс', 'дисторшн', 

In [40]:
print(f"Мой словарь дополнительно покрыл: {len(covered_by_wiki)/len(noize_mc_words_lemmatized)*100:.2}% текста")

Мой словарь дополнительно покрыл: 3.3% текста


Финальный остаток

In [41]:
final_unknown = unknown_words - russian_dict_set

print("Остались после обоих словарей:", len(final_unknown))

Остались после обоих словарей: 1350


In [42]:
final_unknown
print(sorted(final_unknown)[:10])

['а-а', 'а-а-а-а-а-а-а-а-а', 'а-ай', 'авиарежиме', 'автографы-то', 'автопати', 'автотюна', 'автотюном', 'ага-ага', 'адеквате-е-а']


Для современных текстов (на примере Noize MC) словарь Wiktionary показал более заметный эффект.<br>
Дополнительное покрытие составило около 3,3% уникальных словоформ текста, отсутствующих в pymorphy.<br>
В отличие от классической литературы, здесь в список попадают:
* разговорная и сленговая лексика: кайфовый, движ, тусить, хайп, ништяк
* заимствования и англицизмы: баттл, дисс, флоу, лайв, хардкор
* имена собственные и культурные отсылки: киану, коппола, тимати, гомер
* интернет- и медиа-лексика: видос, гифка, комментить, блогинг
* обсценная и разговорная речь: дохуя, ебать, пиздос, хуета
* дефисные конструкции и составные слова: бабл-гам, стендап-комик, хип-хоп-культура

<a id="id8"></a>
## Вывод

Словарь pymorphy значительно больше и глубже. Он содержит миллионы словоформ и хорошо покрывает нормативную лексику. Это делает его удобным для морфологического анализа.

Словарь на основе Wiktionary меньше по размеру, но в нём присутствуют имена собственные, топонимы, заимствования, сленг и редкая лексика, которые часто отсутствуют в pymorphy.

На реальных текстах это даёт практический эффект:

* для классической литературы (Пушкин) дополнительное покрытие небольшое (~0.8%),
* для современных текстов (Noize MC) - уже заметное (~3.3%).

Несмотря на небольшие проценты, именно эти слова чаще всего становятся ложными срабатываниями при поиске “неизвестных” слов.

Таким образом, словарь созданный на основе данных Wiktionary хорошо дополняет словарь pymorphy иможет использоваться в задачах NLP.